# Furniture Recommendation Model Training

This notebook demonstrates the training and evaluation of machine learning models for the furniture recommendation system.

## Models Covered:
1. Content-based recommendation using TF-IDF
2. Semantic similarity using sentence transformers
3. Product classification using embeddings
4. Hybrid recommendation system
5. Performance evaluation and optimization

## Approach:
We'll implement multiple recommendation techniques and combine them for better performance.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Libraries
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, classification_report

# Deep Learning
try:
    from sentence_transformers import SentenceTransformer
    sentence_transformers_available = True
    print("Sentence Transformers available")
except ImportError:
    sentence_transformers_available = False
    print("Sentence Transformers not available - using TF-IDF only")

import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

## 1. Data Loading and Preprocessing

In [ ]:
# Load the dataset
try:
    df = pd.read_csv('../data/sample_furniture_data.csv')
    print(f"Dataset loaded! Shape: {df.shape}")
except FileNotFoundError:
    print("Creating sample dataset for model training...")
    
    # Create comprehensive sample data for training
    np.random.seed(42)
    
    furniture_types = [
        'Modern leather sofa', 'Vintage wooden chair', 'Glass coffee table',
        'Ergonomic office chair', 'Rustic dining table', 'Memory foam mattress',
        'Industrial bar stool', 'Scandinavian nightstand', 'Fabric recliner',
        'Metal bookshelf', 'Wooden dresser', 'Outdoor patio set'
    ]
    
    descriptions = [
        'Luxurious comfortable seating for modern living spaces',
        'Classic handcrafted chair with vintage appeal',
        'Contemporary glass surface with sleek metal legs',
        'Professional office seating with lumbar support',
        'Farmhouse style table perfect for family dining',
        'Premium sleep comfort with cooling technology',
        'Urban loft style seating with adjustable height',
        'Minimalist bedside storage with clean lines',
        'Soft fabric upholstery with reclining function',
        'Space-saving vertical storage solution',
        'Traditional bedroom storage with multiple drawers',
        'Weather-resistant outdoor furniture set'
    ]
    
    # Generate larger dataset
    n_products = 200
    sample_data = {
        'uniq_id': range(1, n_products + 1),
        'title': np.random.choice(furniture_types, n_products),
        'brand': np.random.choice(['IKEA', 'Herman Miller', 'West Elm', 'Pottery Barn', 'CB2', 'Article', 'Wayfair'], n_products),
        'description': np.random.choice(descriptions, n_products),
        'price': np.random.uniform(50, 3000, n_products).round(2),
        'categories': np.random.choice([
            'Living Room > Sofas', 'Living Room > Coffee Tables', 'Living Room > TV Stands',
            'Bedroom > Beds', 'Bedroom > Nightstands', 'Bedroom > Dressers',
            'Office > Chairs', 'Office > Desks', 'Dining Room > Tables', 'Dining Room > Chairs',
            'Outdoor > Patio Sets', 'Storage > Bookcases'
        ], n_products),
        'material': np.random.choice(['Wood', 'Metal', 'Fabric', 'Glass', 'Leather', 'Plastic'], n_products),
        'color': np.random.choice(['Brown', 'Black', 'White', 'Gray', 'Blue', 'Red', 'Green'], n_products)
    }
    
    df = pd.DataFrame(sample_data)
    print(f"Sample dataset created! Shape: {df.shape}")

# Preprocessing
print("\n=== Data Preprocessing ===")

# Handle missing values
df = df.fillna('')

# Convert price to numeric
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# Create combined text for content-based filtering
df['combined_features'] = (
    df['title'].astype(str) + ' ' +
    df['description'].astype(str) + ' ' +
    df['categories'].astype(str) + ' ' +
    df['brand'].astype(str) + ' ' +
    df['material'].astype(str) + ' ' +
    df['color'].astype(str)
)

# Extract primary category
df['primary_category'] = df['categories'].str.split(' > ').str[0]

print(f"Data preprocessed successfully!")
print(f"Features created: {df.columns.tolist()}")
print(f"Combined features sample: {df['combined_features'].iloc[0][:100]}...")

## 2. Content-Based Recommendation Model (TF-IDF)

In [ ]:
print("=== Training TF-IDF Content-Based Model ===")

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    ngram_range=(1, 2),  # Include bigrams
    lowercase=True,
    min_df=2  # Ignore terms that appear in less than 2 documents
)

# Fit and transform the combined features
tfidf_matrix = tfidf.fit_transform(df['combined_features'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(tfidf.vocabulary_)}")

# Calculate cosine similarity matrix
cosine_sim_tfidf = linear_kernel(tfidf_matrix, tfidf_matrix)

print(f"Cosine similarity matrix shape: {cosine_sim_tfidf.shape}")

# Function to get TF-IDF based recommendations
def get_tfidf_recommendations(product_id, cosine_sim=cosine_sim_tfidf, df=df, top_k=5):
    """
    Get product recommendations based on TF-IDF similarity
    
    Args:
        product_id: ID of the product to get recommendations for
        cosine_sim: Cosine similarity matrix
        df: DataFrame with product information
        top_k: Number of recommendations to return
    
    Returns:
        List of recommended product information
    """
    try:
        # Get the index of the product
        idx = df[df['uniq_id'] == product_id].index[0]
        
        # Get similarity scores for all products
        sim_scores = list(enumerate(cosine_sim[idx]))
        
        # Sort by similarity score
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Get top k similar products (excluding the product itself)
        sim_scores = sim_scores[1:top_k+1]
        
        # Get product indices
        product_indices = [i[0] for i in sim_scores]
        
        # Return recommended products with similarity scores
        recommendations = []
        for idx, score in zip(product_indices, [s[1] for s in sim_scores]):
            rec = df.iloc[idx].copy()
            rec['similarity_score'] = score
            recommendations.append(rec)
        
        return recommendations
    
    except IndexError:
        print(f"Product ID {product_id} not found")
        return []

# Test TF-IDF recommendations
test_product_id = df['uniq_id'].iloc[0]
print(f"\nTesting TF-IDF recommendations for product ID: {test_product_id}")
print(f"Original product: {df[df['uniq_id'] == test_product_id]['title'].iloc[0]}")

tfidf_recs = get_tfidf_recommendations(test_product_id, top_k=3)
print("\nTop 3 TF-IDF Recommendations:")
for i, rec in enumerate(tfidf_recs, 1):
    print(f"{i}. {rec['title']} (Score: {rec['similarity_score']:.3f})")

## 3. Semantic Similarity Model (Sentence Transformers)

In [ ]:
if sentence_transformers_available:
    print("=== Training Semantic Similarity Model ===")
    
    # Load pre-trained sentence transformer model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"Loaded model: {model}")
    
    # Generate embeddings for all products
    print("Generating embeddings...")
    embeddings = model.encode(df['combined_features'].tolist(), show_progress_bar=True)
    
    print(f"Embeddings shape: {embeddings.shape}")
    
    # Calculate semantic similarity matrix
    semantic_similarity = cosine_similarity(embeddings)
    print(f"Semantic similarity matrix shape: {semantic_similarity.shape}")
    
    # Function to get semantic-based recommendations
    def get_semantic_recommendations(product_id, similarity_matrix=semantic_similarity, df=df, top_k=5):
        """
        Get product recommendations based on semantic similarity
        """
        try:
            idx = df[df['uniq_id'] == product_id].index[0]
            sim_scores = list(enumerate(similarity_matrix[idx]))
            sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
            sim_scores = sim_scores[1:top_k+1]
            
            recommendations = []
            for idx, score in zip([i[0] for i in sim_scores], [s[1] for s in sim_scores]):
                rec = df.iloc[idx].copy()
                rec['similarity_score'] = score
                recommendations.append(rec)
            
            return recommendations
        
        except IndexError:
            print(f"Product ID {product_id} not found")
            return []
    
    # Test semantic recommendations
    print(f"\nTesting Semantic recommendations for product ID: {test_product_id}")
    semantic_recs = get_semantic_recommendations(test_product_id, top_k=3)
    print("\nTop 3 Semantic Recommendations:")
    for i, rec in enumerate(semantic_recs, 1):
        print(f"{i}. {rec['title']} (Score: {rec['similarity_score']:.3f})")
    
    # Save embeddings for later use
    np.save('../data/product_embeddings.npy', embeddings)
    print("\nEmbeddings saved to '../data/product_embeddings.npy'")
    
else:
    print("Sentence Transformers not available. Skipping semantic similarity model.")
    semantic_similarity = None
    embeddings = None

## 4. Product Clustering for Category Discovery

In [ ]:
print("=== Product Clustering Analysis ===")

# Use embeddings if available, otherwise use TF-IDF
if embeddings is not None:
    clustering_features = embeddings
    feature_type = "Semantic Embeddings"
else:
    clustering_features = tfidf_matrix.toarray()
    feature_type = "TF-IDF Features"

print(f"Using {feature_type} for clustering")
print(f"Feature matrix shape: {clustering_features.shape}")

# Determine optimal number of clusters using elbow method
def find_optimal_clusters(features, max_k=10):
    """
    Find optimal number of clusters using elbow method and silhouette score
    """
    inertias = []
    silhouette_scores = []
    K_range = range(2, min(max_k + 1, len(features) // 2))
    
    for k in K_range:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        kmeans.fit(features)
        inertias.append(kmeans.inertia_)
        silhouette_scores.append(silhouette_score(features, kmeans.labels_))
    
    return K_range, inertias, silhouette_scores

# Find optimal clusters
k_range, inertias, sil_scores = find_optimal_clusters(clustering_features, max_k=8)

# Plot results
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(k_range, inertias, 'bo-')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method for Optimal k')
ax1.grid(True)

ax2.plot(k_range, sil_scores, 'ro-')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score vs Number of Clusters')
ax2.grid(True)

plt.tight_layout()
plt.show()

# Choose optimal k (highest silhouette score)
optimal_k = k_range[np.argmax(sil_scores)]
print(f"\nOptimal number of clusters: {optimal_k}")
print(f"Best silhouette score: {max(sil_scores):.3f}")

# Perform final clustering
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(clustering_features)

# Add cluster labels to dataframe
df['cluster'] = cluster_labels

# Analyze clusters
print(f"\n=== Cluster Analysis ===")
for cluster_id in range(optimal_k):
    cluster_products = df[df['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id}: {len(cluster_products)} products")
    
    # Most common categories in this cluster
    top_categories = cluster_products['primary_category'].value_counts().head(3)
    print(f"  Top categories: {dict(top_categories)}")
    
    # Most common brands
    top_brands = cluster_products['brand'].value_counts().head(3)
    print(f"  Top brands: {dict(top_brands)}")
    
    # Average price
    avg_price = cluster_products['price'].mean()
    print(f"  Average price: ${avg_price:.2f}")

# Visualize clusters using PCA
if clustering_features.shape[1] > 2:
    pca = PCA(n_components=2, random_state=42)
    features_2d = pca.fit_transform(clustering_features)
    
    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(features_2d[:, 0], features_2d[:, 1], c=cluster_labels, cmap='tab10', alpha=0.7)
    plt.colorbar(scatter)
    plt.title(f'Product Clusters Visualization (PCA)\n{feature_type}')
    plt.xlabel(f'First Principal Component (explains {pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'Second Principal Component (explains {pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"\nPCA explained variance: {pca.explained_variance_ratio_.sum():.1%}")

## 5. Hybrid Recommendation System

In [ ]:
print("=== Hybrid Recommendation System ===")

def get_hybrid_recommendations(product_id, df=df, top_k=5, 
                             tfidf_weight=0.4, semantic_weight=0.6):
    """
    Get hybrid recommendations combining TF-IDF and semantic similarity
    
    Args:
        product_id: ID of the product to get recommendations for
        df: DataFrame with product information
        top_k: Number of recommendations to return
        tfidf_weight: Weight for TF-IDF similarity
        semantic_weight: Weight for semantic similarity
    
    Returns:
        List of recommended products with combined scores
    """
    try:
        # Get index of the product
        idx = df[df['uniq_id'] == product_id].index[0]
        
        # Get TF-IDF similarities
        tfidf_similarities = cosine_sim_tfidf[idx]
        
        # Get semantic similarities (if available)
        if semantic_similarity is not None:
            semantic_similarities = semantic_similarity[idx]
        else:
            semantic_similarities = np.zeros_like(tfidf_similarities)
            semantic_weight = 0
            tfidf_weight = 1
        
        # Combine similarities with weights
        hybrid_scores = (tfidf_weight * tfidf_similarities + 
                        semantic_weight * semantic_similarities)
        
        # Get top recommendations
        sim_scores = list(enumerate(hybrid_scores))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Exclude the original product and get top k
        sim_scores = sim_scores[1:top_k+1]
        
        # Create recommendations list
        recommendations = []
        for product_idx, score in sim_scores:
            rec = df.iloc[product_idx].copy()
            rec['hybrid_score'] = score
            rec['tfidf_score'] = tfidf_similarities[product_idx]
            if semantic_similarity is not None:
                rec['semantic_score'] = semantic_similarities[product_idx]
            recommendations.append(rec)
        
        return recommendations
    
    except IndexError:
        print(f"Product ID {product_id} not found")
        return []

# Test hybrid recommendations
print(f"Testing Hybrid recommendations for product ID: {test_product_id}")
print(f"Original product: {df[df['uniq_id'] == test_product_id]['title'].iloc[0]}")

hybrid_recs = get_hybrid_recommendations(test_product_id, top_k=5)

print("\nTop 5 Hybrid Recommendations:")
for i, rec in enumerate(hybrid_recs, 1):
    print(f"{i}. {rec['title']}")
    print(f"   Hybrid Score: {rec['hybrid_score']:.3f} | TF-IDF: {rec['tfidf_score']:.3f}", end="")
    if 'semantic_score' in rec:
        print(f" | Semantic: {rec['semantic_score']:.3f}")
    else:
        print("")
    print(f"   Category: {rec['primary_category']} | Brand: {rec['brand']} | Price: ${rec['price']:.2f}")
    print()

## 6. Model Evaluation and Performance Analysis

In [ ]:
print("=== Model Evaluation ===")

def evaluate_recommendations(df, recommendation_function, sample_size=20):
    """
    Evaluate recommendation quality using various metrics
    
    Args:
        df: DataFrame with product information
        recommendation_function: Function to get recommendations
        sample_size: Number of products to test
    
    Returns:
        Dictionary with evaluation metrics
    """
    
    # Sample products for evaluation
    sample_products = df.sample(min(sample_size, len(df)))['uniq_id'].tolist()
    
    metrics = {
        'category_consistency': [],
        'brand_diversity': [],
        'price_similarity': [],
        'material_consistency': []
    }
    
    for product_id in sample_products:
        # Get original product info
        original = df[df['uniq_id'] == product_id].iloc[0]
        
        # Get recommendations
        recommendations = recommendation_function(product_id, top_k=5)
        
        if len(recommendations) == 0:
            continue
        
        # Category consistency (percentage of recommendations in same category)
        same_category = sum(1 for rec in recommendations 
                          if rec['primary_category'] == original['primary_category'])
        category_consistency = same_category / len(recommendations)
        metrics['category_consistency'].append(category_consistency)
        
        # Brand diversity (percentage of different brands)
        unique_brands = len(set(rec['brand'] for rec in recommendations))
        brand_diversity = unique_brands / len(recommendations)
        metrics['brand_diversity'].append(brand_diversity)
        
        # Price similarity (average relative price difference)
        price_diffs = [abs(rec['price'] - original['price']) / original['price'] 
                      for rec in recommendations if original['price'] > 0]
        if price_diffs:
            avg_price_diff = np.mean(price_diffs)
            price_similarity = max(0, 1 - avg_price_diff)  # Convert to similarity score
            metrics['price_similarity'].append(price_similarity)
        
        # Material consistency
        same_material = sum(1 for rec in recommendations 
                          if rec['material'] == original['material'])
        material_consistency = same_material / len(recommendations)
        metrics['material_consistency'].append(material_consistency)
    
    # Calculate average metrics
    avg_metrics = {}
    for metric, values in metrics.items():
        if values:
            avg_metrics[f'avg_{metric}'] = np.mean(values)
            avg_metrics[f'std_{metric}'] = np.std(values)
    
    return avg_metrics

# Evaluate TF-IDF model
print("Evaluating TF-IDF model...")
tfidf_metrics = evaluate_recommendations(df, get_tfidf_recommendations, sample_size=15)

print("\nTF-IDF Model Performance:")
for metric, value in tfidf_metrics.items():
    if 'avg_' in metric:
        print(f"  {metric.replace('avg_', '').title()}: {value:.3f}")

# Evaluate Semantic model (if available)
if semantic_similarity is not None:
    print("\nEvaluating Semantic model...")
    semantic_metrics = evaluate_recommendations(df, get_semantic_recommendations, sample_size=15)
    
    print("\nSemantic Model Performance:")
    for metric, value in semantic_metrics.items():
        if 'avg_' in metric:
            print(f"  {metric.replace('avg_', '').title()}: {value:.3f}")

# Evaluate Hybrid model
print("\nEvaluating Hybrid model...")
hybrid_metrics = evaluate_recommendations(df, get_hybrid_recommendations, sample_size=15)

print("\nHybrid Model Performance:")
for metric, value in hybrid_metrics.items():
    if 'avg_' in metric:
        print(f"  {metric.replace('avg_', '').title()}: {value:.3f}")

# Create comparison visualization
if semantic_similarity is not None:
    models = ['TF-IDF', 'Semantic', 'Hybrid']
    metrics_data = [tfidf_metrics, semantic_metrics, hybrid_metrics]
else:
    models = ['TF-IDF', 'Hybrid']
    metrics_data = [tfidf_metrics, hybrid_metrics]

# Plot comparison
metric_names = ['category_consistency', 'brand_diversity', 'price_similarity', 'material_consistency']
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for i, metric in enumerate(metric_names):
    values = [metrics[f'avg_{metric}'] for metrics in metrics_data if f'avg_{metric}' in metrics]
    
    if values:
        axes[i].bar(models[:len(values)], values, alpha=0.7)
        axes[i].set_title(f'{metric.replace("_", " ").title()}')
        axes[i].set_ylabel('Score')
        axes[i].set_ylim(0, 1)
        
        # Add value labels on bars
        for j, v in enumerate(values):
            axes[i].text(j, v + 0.01, f'{v:.3f}', ha='center')

plt.tight_layout()
plt.suptitle('Model Performance Comparison', y=1.02, fontsize=14)
plt.show()

## 7. Feature Importance Analysis

In [ ]:
print("=== Feature Importance Analysis ===")

# Analyze TF-IDF feature importance
feature_names = tfidf.get_feature_names_out()
tfidf_mean_scores = np.mean(tfidf_matrix.toarray(), axis=0)

# Get top features
top_indices = np.argsort(tfidf_mean_scores)[-20:]
top_features = [(feature_names[i], tfidf_mean_scores[i]) for i in top_indices]
top_features.sort(key=lambda x: x[1], reverse=True)

print("\nTop 20 TF-IDF Features (by average score):")
for i, (feature, score) in enumerate(top_features, 1):
    print(f"{i:2d}. {feature:20s} {score:.4f}")

# Visualize feature importance
plt.figure(figsize=(12, 8))
features, scores = zip(*top_features)
y_pos = np.arange(len(features))

plt.barh(y_pos, scores, alpha=0.7)
plt.yticks(y_pos, features)
plt.xlabel('Average TF-IDF Score')
plt.title('Top 20 Most Important Features (TF-IDF)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# Analyze categorical feature distributions
print("\n=== Categorical Feature Analysis ===")

categorical_features = ['primary_category', 'brand', 'material', 'color']

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for i, feature in enumerate(categorical_features):
    value_counts = df[feature].value_counts().head(10)
    
    axes[i].bar(range(len(value_counts)), value_counts.values, alpha=0.7)
    axes[i].set_title(f'{feature.replace("_", " ").title()} Distribution')
    axes[i].set_xticks(range(len(value_counts)))
    axes[i].set_xticklabels(value_counts.index, rotation=45, ha='right')
    axes[i].set_ylabel('Count')
    
    # Add value labels
    for j, v in enumerate(value_counts.values):
        axes[i].text(j, v + 0.5, str(v), ha='center')

plt.tight_layout()
plt.show()

## 8. Model Optimization and Hyperparameter Tuning

In [ ]:
print("=== Model Optimization ===")

# Test different TF-IDF parameters
def optimize_tfidf_parameters():
    """
    Test different TF-IDF parameters and evaluate performance
    """
    
    parameter_combinations = [
        {'max_features': 1000, 'ngram_range': (1, 1), 'min_df': 1},
        {'max_features': 3000, 'ngram_range': (1, 2), 'min_df': 2},
        {'max_features': 5000, 'ngram_range': (1, 2), 'min_df': 2},
        {'max_features': 5000, 'ngram_range': (1, 3), 'min_df': 3},
    ]
    
    best_score = 0
    best_params = None
    results = []
    
    print("Testing TF-IDF parameter combinations...")
    
    for i, params in enumerate(parameter_combinations):
        print(f"\nTesting combination {i+1}: {params}")
        
        # Create new TF-IDF vectorizer with these parameters
        test_tfidf = TfidfVectorizer(
            max_features=params['max_features'],
            stop_words='english',
            ngram_range=params['ngram_range'],
            lowercase=True,
            min_df=params['min_df']
        )
        
        # Fit and transform
        test_tfidf_matrix = test_tfidf.fit_transform(df['combined_features'])
        test_cosine_sim = linear_kernel(test_tfidf_matrix, test_tfidf_matrix)
        
        # Create recommendation function for this configuration
        def test_recommendation_func(product_id, top_k=5):
            return get_tfidf_recommendations(product_id, cosine_sim=test_cosine_sim, 
                                           df=df, top_k=top_k)
        
        # Evaluate
        metrics = evaluate_recommendations(df, test_recommendation_func, sample_size=10)
        
        # Calculate composite score (weighted average of metrics)
        composite_score = (
            0.3 * metrics.get('avg_category_consistency', 0) +
            0.2 * metrics.get('avg_brand_diversity', 0) +
            0.3 * metrics.get('avg_price_similarity', 0) +
            0.2 * metrics.get('avg_material_consistency', 0)
        )
        
        results.append({
            'params': params,
            'composite_score': composite_score,
            'metrics': metrics
        })
        
        print(f"  Composite Score: {composite_score:.3f}")
        
        if composite_score > best_score:
            best_score = composite_score
            best_params = params
    
    print(f"\nBest parameters: {best_params}")
    print(f"Best composite score: {best_score:.3f}")
    
    return results, best_params

# Run optimization
optimization_results, best_tfidf_params = optimize_tfidf_parameters()

# Test different hybrid weight combinations
if semantic_similarity is not None:
    print("\n=== Hybrid Model Weight Optimization ===")
    
    weight_combinations = [
        (0.8, 0.2),  # TF-IDF heavy
        (0.6, 0.4),  # TF-IDF moderate
        (0.4, 0.6),  # Balanced
        (0.2, 0.8),  # Semantic heavy
        (0.0, 1.0),  # Semantic only
    ]
    
    best_hybrid_score = 0
    best_weights = None
    
    for tfidf_w, semantic_w in weight_combinations:
        print(f"\nTesting weights: TF-IDF={tfidf_w}, Semantic={semantic_w}")
        
        # Create recommendation function with these weights
        def test_hybrid_func(product_id, top_k=5):
            return get_hybrid_recommendations(product_id, df=df, top_k=top_k,
                                            tfidf_weight=tfidf_w, 
                                            semantic_weight=semantic_w)
        
        # Evaluate
        metrics = evaluate_recommendations(df, test_hybrid_func, sample_size=10)
        
        # Calculate composite score
        composite_score = (
            0.3 * metrics.get('avg_category_consistency', 0) +
            0.2 * metrics.get('avg_brand_diversity', 0) +
            0.3 * metrics.get('avg_price_similarity', 0) +
            0.2 * metrics.get('avg_material_consistency', 0)
        )
        
        print(f"  Composite Score: {composite_score:.3f}")
        
        if composite_score > best_hybrid_score:
            best_hybrid_score = composite_score
            best_weights = (tfidf_w, semantic_w)
    
    print(f"\nBest hybrid weights: TF-IDF={best_weights[0]}, Semantic={best_weights[1]}")
    print(f"Best hybrid score: {best_hybrid_score:.3f}")

## 9. Model Persistence and Export

In [ ]:
print("=== Model Persistence ===")

import pickle
import joblib

# Save TF-IDF model and matrix
print("Saving TF-IDF model...")
joblib.dump(tfidf, '../data/tfidf_vectorizer.pkl')
joblib.dump(cosine_sim_tfidf, '../data/tfidf_cosine_similarity.pkl')
print("TF-IDF model saved successfully")

# Save semantic similarity matrix (if available)
if semantic_similarity is not None:
    print("Saving semantic similarity matrix...")
    np.save('../data/semantic_similarity_matrix.npy', semantic_similarity)
    print("Semantic similarity matrix saved successfully")

# Save clustering model
print("Saving clustering model...")
joblib.dump(kmeans, '../data/kmeans_model.pkl')
print("Clustering model saved successfully")

# Save processed dataset with clusters
print("Saving processed dataset...")
df.to_csv('../data/processed_furniture_data_with_clusters.csv', index=False)
print("Processed dataset saved successfully")

# Create model configuration file
model_config = {
    'tfidf_params': {
        'max_features': tfidf.max_features,
        'ngram_range': tfidf.ngram_range,
        'min_df': tfidf.min_df,
        'vocabulary_size': len(tfidf.vocabulary_)
    },
    'clustering': {
        'n_clusters': optimal_k,
        'algorithm': 'KMeans',
        'random_state': 42
    },
    'dataset_info': {
        'n_products': len(df),
        'n_features': tfidf_matrix.shape[1],
        'categories': df['primary_category'].unique().tolist(),
        'brands': df['brand'].unique().tolist()
    },
    'performance_metrics': {
        'tfidf_metrics': tfidf_metrics,
        'hybrid_metrics': hybrid_metrics
    },
    'optimization_results': {
        'best_tfidf_params': best_tfidf_params
    }
}

if semantic_similarity is not None:
    model_config['semantic_model'] = {
        'model_name': 'all-MiniLM-L6-v2',
        'embedding_dim': embeddings.shape[1]
    }
    model_config['performance_metrics']['semantic_metrics'] = semantic_metrics
    if 'best_weights' in locals():
        model_config['optimization_results']['best_hybrid_weights'] = best_weights

# Save configuration
with open('../data/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2, default=str)

print("\nModel configuration saved to '../data/model_config.json'")

# Print summary
print("\n=== MODEL TRAINING SUMMARY ===")
print(f"✓ TF-IDF model trained with {tfidf_matrix.shape[1]} features")
if semantic_similarity is not None:
    print(f"✓ Semantic model trained with {embeddings.shape[1]}D embeddings")
print(f"✓ Product clustering with {optimal_k} clusters")
print(f"✓ Hybrid recommendation system configured")
print(f"✓ All models saved to '../data/' directory")

print("\nFiles created:")
print("- tfidf_vectorizer.pkl")
print("- tfidf_cosine_similarity.pkl")
if semantic_similarity is not None:
    print("- product_embeddings.npy")
    print("- semantic_similarity_matrix.npy")
print("- kmeans_model.pkl")
print("- processed_furniture_data_with_clusters.csv")
print("- model_config.json")

print("\n🎉 Model training completed successfully!")